# Intro to Cognee — building a Company Brain

## Why a company brain?

Company knowledge is **fragmented by default**. The same context is smeared across
Slack threads, meeting notes, email, docs, and tickets — and no single person has
seen all of it. A bug gets reported in one channel, diagnosed in a second, and
resolved in a meeting nobody else attended. Three months later someone re-asks the
same question because the answer is unfindable.

Meanwhile everyone has wired up their own AI tools — but each one is an island. Your
chatbot knows your prompt, not last week's standup; your meeting notetaker captures
the call, then forgets it. The intelligence is there; the **shared memory** is not.

A *company brain* fixes the memory, not the model: ingest the places people actually
talk into **one knowledge graph**, where people, projects, topics, decisions, and
issues are linked across every source. Then point applications at that graph.

## What one shared graph unlocks

Once the context is unified and queryable, the same graph powers a family of apps:

- **Support agent for repeat questions** — across many client channels, customers
  ask the same things. The graph already holds the prior answer, so an agent can
  respond instead of pinging a human again.
- **Expert finder** — *who* should I nudge about this? The graph knows who has
  spoken to a topic before, so questions route to the right person automatically.
- **Project follow-up agent** — from deadlines, emails, and meeting notes, remind the
  relevant people when an action is due or slipping.
- **Catch-up / onboarding search** — missed a meeting, or not staffed on a project but
  need to know where it stands? Ask the graph instead of reconstructing it by hand.
- **Hierarchy-aware disambiguation** — when facts conflict, resolve them using the
  company's structure (a manager's statement outranks a junior's guess).
- **Permissions across all of the above** — the graph is scoped so each person and
  agent only sees what they're allowed to. (Cognee tags every node with a node set —
  the same mechanism we use in section 3 — which is the foundation access control
  builds on.)

This notebook builds the **foundation** those apps stand on: the unified graph, and
the core `remember → recall → improve` loop. We use a tiny slice of real data — a few
Slack threads and a Granola meeting note — so you can see each idea clearly:

1. **Load data** — `remember()` ingests text *and* builds the graph in one call
2. **Recall** — ask the graph natural-language questions
3. **Node sets** — tag data so you can scope recall to a source/person/project
4. **Custom graph model** — replace generic extraction with a typed schema
5. **Improve** — feedback and self-improvement make the graph sharper over time
6. **Add another source** — drop Granola notes into the same graph

## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)   # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

**`remember()` replaces the old two-step `add()` + `cognify()`** — it ingests the
text, extracts entities/relationships into the graph, *and* (by default) runs a
self-improvement pass. You rarely call `cognify()` directly anymore.

> **Where did `cognify` go?** It still exists. `remember()` = `add()` +
> `cognify()` + self-improvement rolled into one. Reach for the lower-level
> `cognee.add()` + `cognee.cognify(...)` only when you want to **decouple
> ingestion from graph-building** — e.g. bulk-load hundreds of docs and build
> the graph once, re-process existing data with a different schema without
> re-uploading, or run the build as a background job. For everyday use,
> `remember()` is the one call you need.

**The improvement piece.** A company brain is never "done" — new conversations
arrive and earlier answers get corrected. Cognee bakes this in:
`remember(..., self_improvement=True)` (the default) refines the graph on every
write, `cognee.improve()` runs an explicit enrichment pass, and
`recall(..., feedback_influence=...)` lets prior feedback steer future answers.
That's what makes it a *living* graph rather than a one-shot index.

## Setup

You need an LLM key. Put `LLM_API_KEY=sk-...` in the repo's `.env` (Cognee uses
it for extraction and recall). We turn session caching **on** (`CACHING=true`)
because the feedback section (5) needs session memory.

> Run once: `uv pip install -e ".[dev]" jupyterlab ipykernel`

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")   # picks up LLM_API_KEY
os.environ.setdefault("CACHING", "true")  # session memory ON (needed for the feedback section)

import cognee

# sample_data lives at the repo root; handle running from tutorial/ or root
ROOT = Path.cwd()
DATA = ROOT / "sample_data" if (ROOT / "sample_data").exists() else ROOT.parent / "sample_data"
DATASET = "company_brain"

async def reset():
    """Wipe the graph so each section starts from a clean slate."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

import re as _re  # (re already imported above)
def first_speaker(text):
    m = _re.search(r"\[([^,\]]+@[^,\]]+),", text)
    return m.group(1) if m else "unknown"

def answer(results):
    """Pull the natural-language answer out of a recall() result list.
    recall() can return a graph entry (.text), a QA entry (.answer), or a
    context entry (.content) depending on how the query auto-routes."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"


async def show_graph(filename, height=500):
    """Render the current graph to its own HTML file and embed it."""
    await cognee.visualize_graph(str(ROOT / filename))
    from IPython.display import IFrame
    return IFrame(src=filename, width="100%", height=height)


from cognee.api.v1.session import add_feedback, get_session

async def show_session(session_id):
    """Print what's stored in session memory for each Q&A turn."""
    entries = await get_session(session_id)
    if not entries:
        print("  (session empty)"); return
    for e in entries:
        print(f"  Q: {(e.question or '')[:55]}")
        print(f"     feedback_text : {e.feedback_text}")
        print(f"     feedback_score: {e.feedback_score}")
        print(f"     memify_metadata: {e.memify_metadata}")

async def show_node_weights(used_graph_element_ids, limit=6):
    """Print cognee's feedback weights for the graph nodes an answer leaned on."""
    from cognee.infrastructure.databases.graph import get_graph_engine
    node_ids = (used_graph_element_ids or {}).get("node_ids", [])
    g = await get_graph_engine()
    weights = await g.get_node_feedback_weights(node_ids)
    return {k: weights[k] for k in list(weights)[:limit]}

print("sample data:", sorted(p.name for p in DATA.glob("*.txt")))


2026-05-30T16:00:56.230546 [info     ] Deleted old log file: /Users/veljko/.cognee/logs/2026-05-30_17-49-34.log [cognee.shared.logging_utils]



2026-05-30T16:00:56.230985 [info     ] Log file created at: /Users/veljko/.cognee/logs/2026-05-30_19-00-56.log [cognee.shared.logging_utils] log_file=/Users/veljko/.cognee/logs/2026-05-30_19-00-56.log



2026-05-30T16:00:56.231273 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]



2026-05-30T16:00:56.231525 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.0 database_path=/Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases os_info='Darwin 25.3.0 (Darwin Kernel Version 25.3.0: Wed Jan 28 20:51:28 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6041)' python_version=3.12.13 structlog_version=25.5.0



2026-05-30T16:00:56.231730 [info     ] Database storage: /Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases [cognee.shared.logging_utils]



2026-05-30T16:00:56.445000 [info     ] auth posture: authentication=required, multi_tenant=enabled (default (no env vars set)) [get_authenticated_user]


sample data: ['granola.txt', 'slack_1.txt', 'slack_2.txt', 'slack_3.txt']


## 1. Load the Slack data with `remember()`

Our Slack threads are saved as plain `.txt` transcripts in `sample_data/` — one
line per message, `[speaker, timestamp] text`. `remember()` takes raw text,
ingests it, and builds the knowledge graph in a single call.

> **In production** you wouldn't read files — you'd pull live threads from the
> Slack API and normalize them to this same shape (see
> `src/company_brain/sources/slack.py`). Once text is in this shape, the source
> doesn't matter.

In [2]:
# Peek at the raw input first. Every source — Slack here, Granola later — is
# normalized to the same transcript shape: one line per message,
# "[speaker, timestamp] text". This is all cognee needs.
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[1].read_text())   # slack_2: the node-sets bug report

# #brain-project: i noticed that there is a bug in code when you want to call remember with node s
[veljko@topoteretes.com, 2026-05-29T09:02] i noticed that there is a bug in code when you want to call remember with node sets. It keeps extracting everything. Do you have any idea what is it about?
## Reported issue candidates
- [veljko@topoteretes.com, 2026-05-29T09:02] i noticed that there is a bug in code when you want to call remember with node sets. It keeps extracting everything. Do you have any idea what is it about?


In [3]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(f.read_text(), dataset_name=DATASET)
    print("remembered", f.name, "— status:", result.to_dict().get("status"))


2026-05-30T16:00:58.188074 [info     ] Deleted Ladybug database files at /Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases/8696a5a9-30fe-49f3-97aa-63b037a92067/d1ab2c00-f0c6-5d63-9760-b29ac80faeb0.lbug [cognee.shared.logging_utils]



2026-05-30T16:00:58.197681 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:00:58.264823 [info     ] Log file created at: /Users/veljko/.cognee/logs/2026-05-30_19-00-56.log [cognee.shared.logging_utils] log_file=/Users/veljko/.cognee/logs/2026-05-30_19-00-56.log



2026-05-30T16:00:58.265199 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]



2026-05-30T16:00:58.265442 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.0 database_path=/Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases os_info='Darwin 25.3.0 (Darwin Kernel Version 25.3.0: Wed Jan 28 20:51:28 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6041)' python_version=3.12.13 structlog_version=25.5.0


...


2026-05-30T16:02:31.630053 [info     ] Coroutine task started: `index_data_points` [run_tasks_base]



2026-05-30T16:02:31.672630 [info     ] Coroutine task completed: `index_data_points` [run_tasks_base]



2026-05-30T16:02:31.679616 [info     ] Pipeline run completed: `fad6d589-0fe4-5405-a7ce-4c61319a7e7b` [run_tasks_with_telemetry()]



2026-05-30T16:02:31.802962 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


remembered slack_3.txt — status: completed


## 2. Recall — ask the graph

`recall()` runs a graph-aware search and returns a natural-language answer by
default (it auto-routes the query for you). Pass `only_context=True` to get the
raw retrieved context instead of a synthesized answer — useful for seeing *what*
the graph pulled before it's summarized.

Our data: someone reported a bug about `remember` with node sets. Let's ask.

In [4]:
results = await cognee.recall(
    "What bug was reported, and who reported it?",
    datasets=[DATASET],
)
print(answer(results))


2026-05-30T16:02:31.826460 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What bug was reported, and who reported it?' [query_router]



2026-05-30T16:02:33.064905 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 1.13s [cognee.shared.logging_utils]



2026-05-30T16:02:33.065575 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]



2026-05-30T16:02:33.071982 [info     ] ID-filtered retrieval: 26 nodes and 46 edges in 0.01s [cognee.shared.logging_utils]



2026-05-30T16:02:33.072709 [info     ] Graph projection completed: 26 nodes, 46 edges in 0.00s [CogneeGraph]



2026-05-30T16:02:33.073302 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 10, 'connection_count': 10}



2026-05-30T16:02:35.928199 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:02:35.973082 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Bug: when calling remember with node sets it extracts everything instead of the expected subset. Reporter: Veljko (veljko@topoteretes.com).


In [5]:
# Peek under the hood: the raw context the answer was built from
ctx = await cognee.recall(
    "node sets bug",
    datasets=[DATASET],
    only_context=True,
    top_k=3,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:200])


2026-05-30T16:02:35.986873 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='node sets bug' [query_router]



2026-05-30T16:02:36.884222 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.79s [cognee.shared.logging_utils]



2026-05-30T16:02:36.885087 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]



2026-05-30T16:02:36.891574 [info     ] ID-filtered retrieval: 26 nodes and 46 edges in 0.01s [cognee.shared.logging_utils]



2026-05-30T16:02:36.892205 [info     ] Graph projection completed: 26 nodes, 46 edges in 0.00s [CogneeGraph]



2026-05-30T16:02:36.892869 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 3, 'connection_count': 3}



2026-05-30T16:02:36.920674 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:02:36.958828 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


- kind='graph_completion' search_type='GRAPH_COMPLETION' text='Nodes:\nNode: # #brain-project: i noticed that there is... [i, noticed, there]\n__node_content_start__\n# #brain-project: i noticed that th


## 3. Node sets — tag data so you can scope and permission it

**Node sets** are tags you attach at ingest time — `source:slack`, `project:x`,
`speaker:y`. They're how you later **scope** a query ("only answer from Slack")
and how you build **permissions** ("this user only sees these node sets").

You attach them by passing `node_set=[...]`, and you scope a recall with
`node_name=[...]`:

```python
await cognee.remember(text, dataset_name=ds, node_set=["source:slack"])
await cognee.recall(query, datasets=[ds], node_name=["source:slack"], scope=["graph"])
```

> **Heads-up for this demo:** scope **enforcement** is reliable on the Cognee
> server/cloud (where the bot in `BOT.md` runs). In this lightweight *in-process*
> notebook the recall-side filter is inconsistent, so don't be surprised if a
> scoped query here still pulls in everything. The tagging API below is exactly
> what you use in production.

In [6]:
# Tag a couple of sources as we ingest them.
NS_DATASET = "node_sets_demo"
for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack"])
for f in sorted(DATA.glob("granola*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:granola"])
print("tagged Slack as source:slack and the meeting note as source:granola")


2026-05-30T16:02:37.019105 [info     ] Pipeline run started: `8fb4142d-4576-500d-93d2-974f67d022b4` [run_tasks_with_telemetry()]



2026-05-30T16:02:37.025834 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]



2026-05-30T16:02:37.032334 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]



2026-05-30T16:02:37.050736 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]



2026-05-30T16:02:37.057365 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]


...


2026-05-30T16:05:05.149565 [info     ] Coroutine task started: `index_data_points` [run_tasks_base]



2026-05-30T16:05:05.195117 [info     ] Coroutine task completed: `index_data_points` [run_tasks_base]



2026-05-30T16:05:05.202332 [info     ] Pipeline run completed: `d498ccfe-3ecf-5ab3-adb5-48957f57584f` [run_tasks_with_telemetry()]



2026-05-30T16:05:05.330324 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


tagged Slack as source:slack and the meeting note as source:granola


In [7]:
# Scope a recall to one node set. (Reliable on the server; in-process this is
# the API shape rather than a hard guarantee — see the note above.)
scoped = await cognee.recall(
    "What was discussed about node sets?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(scoped))


2026-05-30T16:05:05.352827 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What was discussed about node sets?' [query_router]



2026-05-30T16:05:06.518487 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 1.06s [cognee.shared.logging_utils]



2026-05-30T16:05:06.519106 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]



2026-05-30T16:05:06.531164 [info     ] Graph projection completed: 16 nodes, 90 edges in 0.00s [CogneeGraph]



2026-05-30T16:05:06.531864 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 5, 'connection_count': 10}



2026-05-30T16:05:10.619139 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:05:10.664302 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Node sets were discussed as the argument type passed to remember, and that calling remember with node sets triggers a bug where the code keeps extracting everything instead of the expected subset.


## 4. Custom graph model — extract a typed schema

Default extraction discovers entity types on its own, which can be noisy. When
you know the shape of your domain, give Cognee a **graph model**: a set of
Pydantic `DataPoint` classes. Extraction is then constrained to *your* types —
`Person`, `Message`, `Topic`, `Issue` — with the relationships you defined. Pair
it with a `custom_prompt` that tells the LLM how to fill the schema.

`remember()` accepts both, so this stays a one-call flow. (This is also the case
where you might instead split into `add()` + `cognify(graph_model=...)` if you
wanted to re-process existing data with a new schema without re-uploading it.)

Watch the two graphs below: the **before** is what default extraction
produced (lots of generic, self-discovered types), and the **after** is the
same conversations collapsed onto your `Person` / `Message` / `Issue` schema.

In [8]:
# BEFORE: the graph from default extraction (generic, self-discovered types)
await show_graph("graph_default.html")


2026-05-30T16:05:10.768158 [warning  ] No nodes found in the database [cognee.shared.logging_utils]



2026-05-30T16:05:10.769265 [info     ] Graph visualization saved as /Users/veljko/coding/cognee-companybrain/tutorial/graph_default.html [cognee.shared.logging_utils]



2026-05-30T16:05:10.769532 [info     ] The HTML file has been stored at path: /Users/veljko/coding/cognee-companybrain/tutorial/graph_default.html [cognee.shared.logging_utils]


In [9]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Topic(DataPoint):
    label: str
    metadata: dict = {"index_fields": ["label"]}

class Issue(DataPoint):
    summary: str
    reported_by: Person
    metadata: dict = {"index_fields": ["summary"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    about: Optional[list[Topic]] = None
    reports: Optional[list[Issue]] = None
    metadata: dict = {"index_fields": ["text"]}

class Thread(DataPoint):
    channel: str
    started_at: str
    participants: list[Person]
    messages: list[Message]
    metadata: dict = {"index_fields": ["channel"]}

In [10]:
EXTRACTION_PROMPT = """Extract a company conversation graph from the transcript.
Use the provided Thread graph model with Person, Message, Topic, and Issue nodes.
- Keep every transcript line as a Message with its exact speaker, timestamp, and text.
- Create one Person per email and reuse the same Person across messages.
- If a message reports a bug or broken behavior, create an Issue linked from that Message.
Prefer exact source text over summaries."""

await reset()
for f in slack_files:
    text = f.read_text()
    await cognee.remember(
        text,
        dataset_name=DATASET,
        node_set=["source:slack", f"speaker:{first_speaker(text)}"],
        graph_model=Thread,            # <-- typed schema
        custom_prompt=EXTRACTION_PROMPT,
    )
print("typed graph built")


2026-05-30T16:05:10.956067 [info     ] Deleted Ladybug database files at /Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases/6a8cd5af-de0d-4976-8dd2-ddda9121a942/c2ed8d9a-127a-58a7-9f54-21bbe5a3f997.lbug [cognee.shared.logging_utils]



2026-05-30T16:05:10.965082 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:05:11.071212 [info     ] Deleted Ladybug database files at /Users/veljko/coding/cognee-companybrain/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases/6a8cd5af-de0d-4976-8dd2-ddda9121a942/440e2a1b-b3ed-5223-ba0c-a0ee043ccfd5.lbug [cognee.shared.logging_utils]



2026-05-30T16:05:11.080753 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:05:12.102216 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


...


2026-05-30T16:07:43.382319 [info     ] Coroutine task started: `index_data_points` [run_tasks_base]



2026-05-30T16:07:43.425493 [info     ] Coroutine task completed: `index_data_points` [run_tasks_base]



2026-05-30T16:07:43.432597 [info     ] Pipeline run completed: `531ea1bc-680c-5db2-b8a7-7359f8e8b53d` [run_tasks_with_telemetry()]



2026-05-30T16:07:43.563115 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


typed graph built


In [11]:
# AFTER: the same data, now constrained to the typed Person/Message/Issue schema
await show_graph("graph_typed.html")


2026-05-30T16:07:43.674312 [warning  ] No nodes found in the database [cognee.shared.logging_utils]



2026-05-30T16:07:43.675413 [info     ] Graph visualization saved as /Users/veljko/coding/cognee-companybrain/tutorial/graph_typed.html [cognee.shared.logging_utils]



2026-05-30T16:07:43.675736 [info     ] The HTML file has been stored at path: /Users/veljko/coding/cognee-companybrain/tutorial/graph_typed.html [cognee.shared.logging_utils]


In [12]:
typed = await cognee.recall(
    "Summarize the reported issue and who is involved.",
    datasets=[DATASET],
)
print(answer(typed))


2026-05-30T16:07:43.690698 [info     ] query_router: routed=GRAPH_SUMMARY_COMPLETION score=5.0 query='Summarize the reported issue and who is involved.' scores={'GRAPH_SUMMARY_COMPLETION': 5.0} [query_router]



2026-05-30T16:07:45.971448 [info     ] Vector collection retrieval completed: Retrieved distances from 13 collections in 2.17s [cognee.shared.logging_utils]



2026-05-30T16:07:45.972274 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]



2026-05-30T16:07:45.978918 [info     ] ID-filtered retrieval: 27 nodes and 38 edges in 0.01s [cognee.shared.logging_utils]



2026-05-30T16:07:45.979581 [info     ] Graph projection completed: 27 nodes, 38 edges in 0.00s [CogneeGraph]



2026-05-30T16:07:45.980188 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 15, 'connection_count': 10}



2026-05-30T16:08:13.728840 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:08:13.778178 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Veljko reported a bug in the #brain-project channel (“I noticed that there is a bug...”); exact bug details aren’t specified in the provided context. Milenko (milenko@topoteretes.com) acknowledged the report and offered to investigate and arrange a call to discuss. Messages are in #brain-project; timestamps: veljko’s message ~2026-05-29T08:59, milenko’s reply ~2026-05-29T09:03.


## 5. Feedback & self-improvement — and what changes in each store

A company brain should *learn*. Cognee improves in two ways:

- **Implicit** — `remember(..., self_improvement=True)` (the default) refines
  the graph on every write. You've had this in every cell above.
- **Explicit feedback loop** — `recall` → `add_feedback` → `improve` → `recall`.
  This teaches the brain things it can't extract from text, like *who owns what*.

We'll run that loop and **watch both stores change** at each step:

| Store | Inspect with | What `improve()` does to it |
|---|---|---|
| Session memory | `get_session()` | records your feedback, then flips `feedback_weights_applied` to `True` |
| Graph | `get_node_feedback_weights()` | shifts the feedback weight on the nodes the answer used |

Feedback needs session memory, so the setup cell sets `CACHING=true` and we pass
a `session_id` to `recall`. Scores are **1 (bad) … 5 (great)**.

In [13]:
# BEAT 0 — a small, self-contained graph for this lesson, then snapshot BOTH stores.
# (Default extraction in its own dataset, so the answer is concrete and node weights
#  start at the default 0.5 — independent of the typed graph above.)
FB_DATASET = "feedback_demo"
SESSION = "feedback-demo"
for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=FB_DATASET)

Q = "Who is the go-to expert for node set problems?"
before = await cognee.recall(Q, datasets=[FB_DATASET], session_id=SESSION)
print("ANSWER (before):", answer(before))

qa = (await get_session(SESSION))[-1]          # the Q&A we just created
used = qa.used_graph_element_ids               # graph nodes/edges this answer used

print("\n— SESSION MEMORY (fresh turn) —");      await show_session(SESSION)
print("\n— GRAPH weights BEFORE —", await show_node_weights(used))


2026-05-30T16:08:13.839409 [info     ] Pipeline run started: `79309642-8aee-5f4b-94b7-fb1a6203b6b2` [run_tasks_with_telemetry()]



2026-05-30T16:08:13.846227 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]



2026-05-30T16:08:13.852867 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]



2026-05-30T16:08:13.870459 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]



2026-05-30T16:08:13.877030 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]


...


2026-05-30T16:09:48.440635 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 9, 'connection_count': 10}



2026-05-30T16:09:52.249805 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:09:52.293172 [info     ] recall: 1 results across sources=['session', 'graph'] (session=feedback-demo) [recall]


ANSWER (before): No go-to expert is specified in the provided context. The only person mentioned is the reporter: veljko@topoteretes.com.

— SESSION MEMORY (fresh turn) —
  Q: Who is the go-to expert for node set problems?
     feedback_text : None
     feedback_score: None
     memify_metadata: None



— GRAPH weights BEFORE — {'4d59c4c0-2fb5-5aa0-b756-56e6442f8bbd': 0.5, 'd5702277-e594-5b80-88e2-dfb0b185c0e0': 0.5, '1b5ec617-de93-5d04-ad6b-2c3200401a66': 0.5, '0db3e7b1-fa4c-5425-85bc-015895557e16': 0.5, '4e0d81b6-0bd5-57ff-ae4e-5111714c3bdc': 0.5, 'dffcc39f-2b68-59bc-962f-7dcfa5c2c9ba': 0.5}


In [14]:
# BEAT 1 — a teammate corrects it. Only SESSION MEMORY changes here.
await add_feedback(
    SESSION, qa.qa_id,
    feedback_text="Wrong owner. Milenko is the expert for node sets — route these to him.",
    feedback_score=1,                           # 1 = bad answer (scale is 1..5)
)
print("— SESSION MEMORY (feedback recorded, not yet applied) —")
await show_session(SESSION)
# note: memify_metadata is now {'feedback_weights_applied': False}

— SESSION MEMORY (feedback recorded, not yet applied) —
  Q: Who is the go-to expert for node set problems?
     feedback_text : Wrong owner. Milenko is the expert for node sets — route these to him.
     feedback_score: 1
     memify_metadata: {'feedback_weights_applied': False}


In [15]:
# BEAT 2 — improve() folds the feedback in. Now BOTH stores change.
await cognee.improve(dataset=FB_DATASET, session_ids=[SESSION], feedback_alpha=0.8)

print("— SESSION MEMORY (applied) —");            await show_session(SESSION)
print("\n— GRAPH weights AFTER —", await show_node_weights(used))
# memify_metadata flips to True; the used nodes' weights drop 0.5 -> 0.1
# (they produced an answer the feedback rated low)


2026-05-30T16:09:52.442837 [info     ] Dataset 832503c2-4199-5e1c-a392-b1faa0320e45 is already processed. [cognee.modules.pipelines.layers.check_pipeline_run_qualification]



2026-05-30T16:09:52.451587 [info     ] Pipeline run started: `ff04c05a-e7ea-5deb-a523-ab8b63670d90` [run_tasks_with_telemetry()]



2026-05-30T16:09:52.458357 [info     ] Async Generator task started: `extract_feedback_qas` [run_tasks_base]



2026-05-30T16:09:52.464831 [info     ] Extracting feedback QAs from session cache [extract_feedback_qas]



2026-05-30T16:09:52.465224 [info     ] Coroutine task started: `apply_feedback_weights` [run_tasks_base]


...


2026-05-30T16:10:18.615913 [info     ] sync_graph_to_session: dataset=feedback_demo since=beginning [sync_graph_to_session]



2026-05-30T16:10:18.619193 [info     ] sync_graph_to_session: no new edges to sync [sync_graph_to_session]



2026-05-30T16:10:18.619473 [info     ] improve: synced 0 edges to session 'feedback-demo' [improve]


— SESSION MEMORY (applied) —
  Q: Who is the go-to expert for node set problems?
     feedback_text : Wrong owner. Milenko is the expert for node sets — route these to him.
     feedback_score: 1
     memify_metadata: {'feedback_weights_applied': True}



— GRAPH weights AFTER — {'4d59c4c0-2fb5-5aa0-b756-56e6442f8bbd': 0.5, 'd5702277-e594-5b80-88e2-dfb0b185c0e0': 0.1, '1b5ec617-de93-5d04-ad6b-2c3200401a66': 0.1, '0db3e7b1-fa4c-5425-85bc-015895557e16': 0.1, '4e0d81b6-0bd5-57ff-ae4e-5111714c3bdc': 0.1, 'dffcc39f-2b68-59bc-962f-7dcfa5c2c9ba': 0.1}


In [16]:
# BEAT 3 — re-ask. The feedback now influences retrieval ranking.
after = await cognee.recall(Q, datasets=[FB_DATASET], session_id=SESSION, feedback_influence=1.0)
print("ANSWER (after):", answer(after))
# On a corpus this tiny the top answer may not move yet — but the weights that
# drive future ranking have changed, which is what you just saw in the graph store.


2026-05-30T16:10:18.739520 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='Who is the go-to expert for node set problems?' [query_router]



2026-05-30T16:10:19.905039 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 1.15s [cognee.shared.logging_utils]



2026-05-30T16:10:19.906045 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]



2026-05-30T16:10:19.910201 [info     ] ID-filtered retrieval: 39 nodes and 69 edges in 0.00s [cognee.shared.logging_utils]



2026-05-30T16:10:19.911205 [info     ] Graph projection completed: 39 nodes, 69 edges in 0.00s [CogneeGraph]



2026-05-30T16:10:19.911906 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 8, 'connection_count': 10}



2026-05-30T16:10:22.843505 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:10:22.892973 [info     ] recall: 2 results across sources=['session', 'graph'] (session=feedback-demo) [recall]


ANSWER (after): No go-to expert is specified in the provided context. The only person mentioned is the reporter: veljko@topoteretes.com.


**What you just saw** — the proof is in the two stores, not the answer text:

- **Beat 1** touched *only* session memory: `feedback_text`/`feedback_score` got
  recorded, but `feedback_weights_applied` was still `False`.
- **Beat 2** is where `improve()` did its work: it flipped that flag to `True` and
  pushed the feedback into the **graph**, dropping the feedback weight on the nodes
  behind the low-rated answer (`0.5 → 0.1`). *That* is the improvement.

The recalled answer may not visibly change on a 3-message corpus (there's no
"Milenko is the expert" fact to surface, and one downvote can't flip a single
candidate). Over real usage these weights re-rank retrieval toward answers people
validated. Knobs: `feedback_score` (1–5), `feedback_alpha`, `feedback_influence`.

## 6. Add another source — Granola meeting notes

The payoff. The bug was *reported* in Slack, but it was *resolved* in a meeting
captured in a Granola note. Because Granola notes normalize to the same
transcript shape, adding them is the same `remember()` call with the same schema.
The note joins the **existing** graph, so a single question can span both sources.

In [17]:
granola_files = sorted(DATA.glob("granola*.txt"))
for f in granola_files:
    await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:granola"],
        graph_model=Thread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)


2026-05-30T16:10:22.949841 [info     ] Pipeline run started: `bf94600e-b813-5285-bbd2-11bed5cbd929` [run_tasks_with_telemetry()]



2026-05-30T16:10:22.956907 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]



2026-05-30T16:10:22.963538 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]



2026-05-30T16:10:22.982623 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]



2026-05-30T16:10:22.989387 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]


...


2026-05-30T16:11:20.821137 [info     ] Coroutine task started: `index_data_points` [run_tasks_base]



2026-05-30T16:11:20.864663 [info     ] Coroutine task completed: `index_data_points` [run_tasks_base]



2026-05-30T16:11:20.871776 [info     ] Pipeline run completed: `531ea1bc-680c-5db2-b8a7-7359f8e8b53d` [run_tasks_with_telemetry()]



2026-05-30T16:11:20.995795 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


remembered granola.txt


In [18]:
# This answer requires connecting the Slack bug report to the Granola resolution
cross = await cognee.recall(
    "What was the fix for the node sets bug, and where was it explained?",
    datasets=[DATASET],
)
print(answer(cross))

# Node sets work across sources too — scope this query to just the Granola notes:
granola_now = await cognee.recall(
    "What bug was discussed?",
    datasets=[DATASET],
    node_name=["source:granola"],
    scope=["graph"],
)
print("scoped to source:granola ->", answer(granola_now))


2026-05-30T16:11:21.017664 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What was the fix for the node sets bug, and where was it explained?' [query_router]



2026-05-30T16:11:22.184133 [info     ] Vector collection retrieval completed: Retrieved distances from 14 collections in 1.06s [cognee.shared.logging_utils]



2026-05-30T16:11:22.184859 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]



2026-05-30T16:11:22.191508 [info     ] ID-filtered retrieval: 43 nodes and 63 edges in 0.01s [cognee.shared.logging_utils]



2026-05-30T16:11:22.192300 [info     ] Graph projection completed: 43 nodes, 63 edges in 0.00s [CogneeGraph]


...


2026-05-30T16:11:29.909422 [info     ] Graph projection completed: 3 nodes, 6 edges in 0.00s [CogneeGraph]



2026-05-30T16:11:29.909920 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 3, 'connection_count': 6}



2026-05-30T16:11:35.112022 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]



2026-05-30T16:11:35.160594 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


scoped to source:granola -> A bug in handling node sets: node set names were being passed instead of node set IDs, causing incorrect search/results. The fix is to pass node set IDs (not names).


## See the graph

`visualize_graph()` writes an interactive HTML view of the whole graph.

In [19]:
await cognee.visualize_graph(str(ROOT / "graph.html"))
from IPython.display import IFrame
IFrame(src="graph.html", width="100%", height=600)


2026-05-30T16:11:35.165787 [warning  ] No nodes found in the database [cognee.shared.logging_utils]



2026-05-30T16:11:35.166661 [info     ] Graph visualization saved as /Users/veljko/coding/cognee-companybrain/tutorial/graph.html [cognee.shared.logging_utils]



2026-05-30T16:11:35.166904 [info     ] The HTML file has been stored at path: /Users/veljko/coding/cognee-companybrain/tutorial/graph.html [cognee.shared.logging_utils]


## Recap & where to go next

You built a company brain with one call, repeated as you layered on structure:

| Concept | Call |
|---|---|
| Load data + build graph | `cognee.remember(text, dataset_name=...)` |
| Ask questions | `cognee.recall(query, datasets=[...])` |
| Scope recall | `remember(..., node_set=[...])` + `recall(..., node_name=[...], scope=["graph"])` |
| Typed extraction | `remember(..., graph_model=Thread, custom_prompt=...)` |
| Improve / feedback | `add_feedback(...)` + `cognee.improve(dataset=..., feedback_alpha=...)` |
| New source | same `remember()` call, different input |
| Decouple ingest/build | drop to `cognee.add()` + `cognee.cognify(...)` |

From here, the full project (`src/company_brain/`) shows the product version:
live Slack/Granola ingestion, an LLM classifier that auto-tags threads, and a
Slack bot that answers from the graph via `recall()`.